In [129]:
import pandas as pd
from IPython.display import display, clear_output
import ipywidgets as widgets
import json

data_loc = "../data/synthetic/components"

### Set up dropdowns

In [130]:
with open(f'{data_loc}/motherboards.json', 'r') as file:
    mobo_data = json.load(file)

mobo_choices = [(f"{mobo['brand']} {mobo['model']}", mobo['id']) for mobo in mobo_data]

In [131]:
# everyone but mobo has id + name
# function time baby

def get_component_choices(component):
    with open(f'{data_loc}/{component}.json', 'r') as file:
        component_data = json.load(file)
    return [(f"{item['name']}", item['id']) for item in component_data]

In [132]:
# also all components need the same stuff for the dropdowns, so let's make a function for that too

def create_dropdown(component, choices=None, display_it=False):
    choices = choices if choices is not None else get_component_choices(component)
    dropdown = widgets.Select(
        options=choices,
        value=choices[0][1],
        description=f'{component}:',
        disabled=False,
        layout=widgets.Layout(width='max-content'),
        rows=len(choices),
    )
    if display_it == True: display(dropdown)
    return dropdown

# note to future self: mobo_dropdown.value to get selected ID, mobo_dropdown.label to get selected label

In [133]:
mobo_dropdown = create_dropdown('motherboards', mobo_choices)

component_list = ["cases", "coolers", "cpus", "fans", "gpus", "lighting", "psus", "ram", "storage"]
dropdowns = [create_dropdown(component) for component in component_list]
dropdowns.insert(0, mobo_dropdown)

I want to check whether components are compatible. There are a couple of ways to do this. I could poll with every selection, or I can just provide a big fat button so you submit the build, then check. 

Performance wise, the latter wins, but I don't think any of this is performance intensive enough to warant a hack. 

Ok, gonna go for polling. 

I also think that gives more opportunity for learning.

## Compatibility checking


In [134]:
### HELPERS 

# loads our entire catalog from the data folder
CATEGORIES = ['cases', 'coolers', 'cpus', 'fans', 'gpus', 'lighting', 'motherboards', 'psus', 'ram', 'storage']
CATALOG = {}
for category in CATEGORIES:
    with open(f'{data_loc}/{category}.json', 'r') as file:
        cat_data = json.load(file)
    CATALOG[category] = cat_data

# helper to grab all of a part's data from the catalog, looking up via ID.
def get_part_by_ID(component, id):
    return next((item for item in CATALOG[component] if item["id"] == id), None)

# I'll need a way to grab the current build
def get_current_build():
    # makes a dictionary with this shape:
    #   { <component type>: <component details from catalog>, ... }
    return { dropdown.description[:-1]: get_part_by_ID(dropdown.description[:-1], dropdown.value) for dropdown in dropdowns }


def get_mobo_name(mobo_data):
    return f"{mobo_data["brand"]} {mobo_data["model"]}"


### case check

In [135]:
def check_case(build):
    case_problems = {}
    case = build["cases"]
    # check that gpu length is supported
    gpu = build["gpus"]
    if gpu["length_mm"] > case["gpu_max_length_mm"]:
        case_problems["gpu_max_length_mm"] = (
            f"{gpu['name']} GPU length ({gpu['length_mm']} mm) exceeds the {case['name']} case limit ({case['gpu_max_length_mm']} mm)."
        )

    # check that cpu height is supported
    cooler = build["coolers"]
    if cooler["type"] == "air" and cooler["height_mm"] > case["cpu_cooler_max_height_mm"]:
        case_problems["cpu_cooler_max_height_mm"] = (
            f"{cooler['name']} cooler height ({cooler['height_mm']} mm) exceeds the {case['name']} case limit ({case['cpu_cooler_max_height_mm']} mm)."
        )

    # check that psu form factor is supported
    psu = build["psus"]
    if psu["form_factor"] not in case["supported_psu_form_factors"]:
        case_problems["supported_psu_form_factors"] = (
            f"{psu['name']} PSU form factor ({psu['form_factor']}) is not supported by the {case['name']} case."
        )

    # check that radiator size is supported
    if cooler["type"] == "liquid" and cooler["radiator_mm"] not in case["radiator_sizes_mm"]:
        case_problems["radiator_sizes_mm"] = (
            f"{cooler['name']} radiator size ({cooler['radiator_mm']} mm) is not supported by the {case['name']} case."
        )

    # check that sata 2.5 is supported
    storage = build["storage"]
    if storage["form_factor"] == "SATA_2.5" and case["sata_bays_2_5"] < 1:
        case_problems["sata_bays_2_5"] = (
            f"{case['name']} case does not support 2.5-inch SATA drives."
        )

    # check that sata 3.5 is supported
    if storage["form_factor"] == "SATA_3.5" and case["sata_bays_3_5"] < 1:
        case_problems["sata_bays_3_5"] = (
            f"{case['name']} case does not support 3.5-inch SATA drives."
        )
        
    return len(case_problems.keys()), case_problems

### cooler check

In [136]:
def check_cooler(build):
    cooler_problems = {}
    cooler = build["coolers"]
    cpu = build["cpus"]

    # check that cooler supports cpu power
    if cooler["cooling_capacity_w"] < cpu["tdp_w"]:
        cooler_problems["cooling_capacity_w"] = (
            f"{cooler['name']} cooling capacity ({cooler['cooling_capacity_w']} W) is below the {cpu['name']} CPU TDP ({cpu['tdp_w']} W)."
        )

    return len(cooler_problems.keys()), cooler_problems

### gpu checks

In [137]:
def check_gpu(build):
    gpu_problems = {}
    gpu = build["gpus"]
    psu = build["psus"]
    
    # check that psu supports gpu power connectors
    for connector, required_count in gpu["power_connectors"].items():
        available_count = psu["connectors"].get(connector, 0)
        if required_count > available_count:
            gpu_problems[connector] = (
                f"{gpu['name']} requires {required_count} {connector} power connector(s), but the {psu['name']} PSU has {available_count}."
            )

    return len(gpu_problems.keys()), gpu_problems

### mobo checks

In [138]:
def case_mobo_compat(case, mobo):
    case_problems = {}
    # check that case supports mobo form factor
    if mobo["form_factor"] not in case["motherboard_form_factors"]:
        case_problems["motherboard_form_factors"] = f"{get_mobo_name(mobo)} motherboard form factor ({mobo['form_factor']}) is not supported by the {case['name']} case."
    return case_problems
    
def cpu_mobo_compat(cpu, mobo):
    cpu_problems = {}
    # check that mobo supports cpu socket
    if cpu["socket"] != mobo["cpu_socket"]:
        cpu_problems["cpu_socket"] = f"{cpu['name']} CPU socket ({cpu['socket']}) is not supported by the {get_mobo_name(mobo)} motherboard ({mobo['cpu_socket']})."
    return cpu_problems

def cooler_mobo_compat(cooler, mobo):
    cooler_problems = {}
    # check that mobo supports cooler mount
    if mobo["cooler_mount"] not in cooler["supported_mounts"]:
        cooler_problems["cooler_mount"] = f"{cooler['name']} does not support the {mobo['cooler_mount']} cooler mount on the {get_mobo_name(mobo)} motherboard."

    # check that mobo supports cooler fan connector
    fan_headers = [
        connector["interface"]
        for connector in mobo["connector_layout"]["connectors"]
        if connector["kind"] == "fan_header"
    ]
    if cooler["fan_connector"] not in fan_headers:
        cooler_problems["cooler_fan_connector"] = f"{cooler['name']} fan connector ({cooler['fan_connector']}) is not supported by the {get_mobo_name(mobo)} motherboard."
    return cooler_problems

def fan_mobo_compat(fan, mobo):
    fan_problems = {}
    # check that mobo supports fan connector
    fan_headers = [
        connector["interface"]
        for connector in mobo["connector_layout"]["connectors"]
        if connector["kind"] == "fan_header"
    ]
    if fan["fan_connector"] not in fan_headers:
        fan_problems["fan_connector"] = f"{fan['name']} fan connector ({fan['fan_connector']}) is not supported by the {get_mobo_name(mobo)} motherboard."
    return fan_problems

def gpu_mobo_compat(gpu, mobo):
    gpu_problems = {}
    # check that mobo supports gpu slot
    pcie_slots = [
        connector["interface"]
        for connector in mobo["connector_layout"]["connectors"]
        if connector["kind"] == "pcie_slot"
    ]
    if gpu["pcie_slot"] not in pcie_slots:
        gpu_problems["pcie_slot"] = f"{gpu['name']} GPU slot ({gpu['pcie_slot']}) is not supported by the {get_mobo_name(mobo)} motherboard."
    return gpu_problems

def ram_mobo_compat(ram, mobo):
    ram_problems = {}
    # check that mobo supports ram type
    if ram["type"] != mobo["memory"]["type"]:
        ram_problems["memory_type"] = f"{ram['name']} memory type ({ram['type']}) is not supported by the {get_mobo_name(mobo)} motherboard ({mobo['memory']['type']})."

    # check that mobo supports ram form factor
    if ram["form_factor"] != mobo["memory"]["form_factor"]:
        ram_problems["memory_form_factor"] = f"{ram['name']} memory form factor ({ram['form_factor']}) is not supported by the {get_mobo_name(mobo)} motherboard ({mobo['memory']['form_factor']})."

    # check that mobo supports ram pin count
    if ram["pins"] != mobo["memory"]["pins"]:
        ram_problems["memory_pins"] = f"{ram['name']} memory pin count ({ram['pins']}) does not match the {get_mobo_name(mobo)} motherboard ({mobo['memory']['pins']})."

    # check that mobo has enough ram slots
    if ram["modules"] > mobo["memory"]["slots"]:
        ram_problems["memory_slots"] = f"{ram['name']} requires {ram['modules']} memory slots, but the {get_mobo_name(mobo)} motherboard has {mobo['memory']['slots']}."

    # check that mobo supports ram capacity
    ram_capacity = ram["modules"] * ram["gb_per_module"]
    if ram_capacity > mobo["memory"]["max_capacity_gb"]:
        ram_problems["memory_capacity_gb"] = f"{ram['name']} capacity ({ram_capacity} GB) exceeds the {get_mobo_name(mobo)} motherboard limit ({mobo['memory']['max_capacity_gb']} GB)."
    return ram_problems

def storage_mobo_compat(storage, mobo):
    storage_problems = {}
    # check that mobo supports storage
    connectors = mobo["connector_layout"]["connectors"]
    # M2
    if storage["form_factor"] == "M2":
        compatible_m2_slots = [
            connector
            for connector in connectors
            if connector["kind"] == "m2_slot"
            and connector["key"] in storage["key"].split("+")
            and storage["size"] in connector["lengths"]
            and storage["protocol"] in connector["protocols"]
        ]
        if not compatible_m2_slots:
            storage_problems["storage"] = f"{storage['name']} is not supported by the {get_mobo_name(mobo)} motherboard's M.2 slots."
    # SATA
    elif storage["form_factor"] in ("SATA_2.5", "SATA_3.5"):
        sata_ports = [
            connector
            for connector in connectors
            if connector["kind"] == "sata_port"
            and connector["interface"] == "SATA_6GBPS"
        ]
        if not sata_ports:
            storage_problems["storage"] = f"The {get_mobo_name(mobo)} motherboard does not support SATA storage."
    return storage_problems

def psu_mobo_compat(mobo, psu):
    psu_problems = {}
    # check that psu supports mobo power connectors
    power_headers = [
        connector["interface"]
        for connector in mobo["connector_layout"]["connectors"]
        if connector["kind"] == "power_header"
    ]
    for connector in set(power_headers):
        required_count = power_headers.count(connector)
        available_count = psu["connectors"].get(connector, 0)
        if required_count > available_count:
            psu_problems[connector] = f"The {get_mobo_name(mobo)} motherboard requires {required_count} {connector} power connector(s), but the {psu['name']} PSU has {available_count}."
    return psu_problems

def check_mobo(build):
    mobo = build["motherboards"]
    mobo_problems = {}
    
    mobo_problems.update(case_mobo_compat(build["cases"], mobo))
    mobo_problems.update(cpu_mobo_compat(build["cpus"], mobo))
    mobo_problems.update(cooler_mobo_compat(build["coolers"], mobo))
    mobo_problems.update(fan_mobo_compat(build["fans"], mobo))
    mobo_problems.update(gpu_mobo_compat(build["gpus"], mobo))
    mobo_problems.update(ram_mobo_compat(build["ram"], mobo))
    mobo_problems.update(storage_mobo_compat(build["storage"], mobo))
    mobo_problems.update(psu_mobo_compat(mobo, build["psus"]))

    return len(mobo_problems.keys()), mobo_problems

### power check

In [ ]:
def storage_power_compat(storage, psu):
    storage_problems = {}
    storage_parts = storage if isinstance(storage, list) else [storage]

    # counts how many SATA connectors are needed (will support multi check if/when that is implemented)
    required_count = sum(1 for part in storage_parts if part["form_factor"] in ("SATA_2.5", "SATA_3.5"))

    # check that psu supports storage power connector
    available_count = psu["connectors"].get("SATA_POWER", 0)
    if required_count > available_count:
        storage_problems["SATA_POWER"] = (
            f"Selected SATA storage requires {required_count} SATA_POWER connector(s), but the {psu['name']} PSU has {available_count}."
        )
    return storage_problems

def build_power_compat(build):
    power_req_problems = {}
    psu = build["psus"]
    storage_parts = build["storage"] if isinstance(build["storage"], list) else [build["storage"]]
    
    # add up wattage needed
    total_power = (
        build["cpus"]["tdp_w"]
        + build["gpus"]["tdp_w"]
        + build["fans"]["power_w"]
        + build["lighting"]["power_w"]
        + sum(storage["power_w"] for storage in storage_parts)
    )

    # check that psu supports required wattage
    if total_power > psu["watts"]:
        power_req_problems["watts"] = (
            f"Build power requirements ({total_power} W) exceed the {psu['name']} PSU capacity ({psu['watts']} W)."
        )
    return power_req_problems

def check_power(build):
    power_problems = {}
    power_problems.update(build_power_compat(build))
    power_problems.update(storage_power_compat(build["storage"], build["psus"]))
    return len(power_problems.keys()), power_problems

### BUILD CHECK
calls all component helpers

In [140]:
# checks if the current selections make a compatible build
build_check_output = widgets.Output()

def check_build(change):
    build = get_current_build()

    # TODO refactor; all power capacity stuff in a single power checker, all case (fit) stuff to a case checker
    compatibility_checks = [
        ("Case", check_case),
        ("Cooler", check_cooler),
        ("GPU", check_gpu),
        ("Motherboard", check_mobo),
        ("Power", check_power),
    ]

    results = [(name, checker(build)) for name, checker in compatibility_checks]
    total_issues = sum(issue_count for _, (issue_count, _) in results)

    with build_check_output:
        clear_output(wait=True)
        # TODO format this nicer
        if total_issues:
            for name, (issue_count, issues) in results:
                if issue_count:
                    print(f"{name} compatibility issues ({issue_count}):")
                    for issue in issues.values():
                        print(f"- {issue}")
        else:
            print("Build is compatible!")

### Set up observers for dropdown choices

In [141]:
# add an observer for every dropdown
def update_build_text(change):
    build_text.value = f"{newline.join([dropdown.value for dropdown in dropdowns])}"

for dropdown in dropdowns:
    dropdown.observe(update_build_text, names="value")
    dropdown.observe(check_build, names="value")

### Display

In [142]:
newline = "\n"
build_text = widgets.Textarea(
    value=f"{newline.join([dropdown.value for dropdown in dropdowns])}",
    placeholder='Type something',
    description='Build:',
    disabled=True,
    rows=len(dropdowns),
    layout=widgets.Layout(justify_content='flex-end', width='max-content')
)

In [143]:
widgets.GridBox(
    children=dropdowns + [build_text, build_check_output],
    layout=widgets.Layout(
        # width='50%',
        grid_template_columns='auto auto auto',
        grid_template_rows='auto auto auto auto', 
        grid_gap='5px 10px',
    )
)

GridBox(children=(Select(description='motherboards:', layout=Layout(width='max-content'), options=(('Fictional…